In [28]:
import re
import random
import pandas as pd
import numpy as np
import csv
from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from difflib import get_close_matches
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [29]:
training = pd.read_csv('/content/Training.csv')
testing = pd.read_csv('/content/Testing.csv')

In [30]:
training.columns = training.columns.str.replace(r"\.\d+$", "", regex=True)
testing.columns = testing.columns.str.replace(r"\.\d+$", "", regex=True)
training = training.loc[:, ~training.columns.duplicated()]
testing = testing.loc[:, ~testing.columns.duplicated()]

In [31]:
cols = training.columns[:-1]
x = training[cols]
y = training['prognosis']

In [32]:
le = preprocessing.LabelEncoder()
y = le.fit_transform(y)

In [33]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=42)

In [34]:
model = RandomForestClassifier(n_estimators=300, random_state=42)
model.fit(x_train, y_train)

RandomForestClassifier(n_estimators=300, random_state=42)

In [35]:
severityDictionary = {}
description_list = {}
precautionDictionary = {}
symptoms_dict = {symptom: idx for idx, symptom in enumerate(x)}

In [36]:
def getDescription():
    with open('/content/symptom_Description.csv') as csv_file:
        for row in csv.reader(csv_file):
            description_list[row[0]] = row[1]

In [37]:
def getSeverityDict():
    with open('/content/Symptom_severity.csv') as csv_file:
        for row in csv.reader(csv_file):
            try:
                severityDictionary[row[0]] = int(row[1])
            except:
                pass

In [38]:
def getprecautionDict():
    with open('/content/symptom_precaution.csv') as csv_file:
        for row in csv.reader(csv_file):
            precautionDictionary[row[0]] = [row[1], row[2], row[3], row[4]]

In [59]:
symptom_synonyms = {
    "stomach ache": "stomach_pain",
    "stomach pain": "stomach_pain",
    "belly pain": "stomach_pain",
    "tummy pain": "stomach_pain",
    "loose motion": "diarrhea",
    "diarrhea": "diarrhea",
    "motions": "diarrhea",
    "high temperature": "fever",
    "temperature": "fever",
    "feaver": "fever",
    "fever": "fever",
    "coughing": "cough",
    "cough":"cough",
    "throat pain": "sore_throat",
    "sore throat": "sore_throat",
    "headache": "headache",
    "nausea": "nausea",
    "vomiting": "vomiting",
    "cold": "chills",
    "chills": "chills",
    "chest pain": "chest_pain",
    "shortness of breath": "breathlessness",
    "breathing issue": "breathlessness",
    "body ache": "muscle_pain",
    "muscle pain": "muscle_pain",
    "fatigue": "fatigue",
    "joint pain": "joint_pain"
}

In [60]:
def extract_symptoms(user_input, all_symptoms):
    extracted = []
    text = user_input.lower().replace("-", " ")
    for phrase, mapped in symptom_synonyms.items():
        if phrase in text:
            extracted.append(mapped)

    for symptom in all_symptoms:
        if symptom.replace("_", " ") in text:
            extracted.append(symptom)

    words = re.findall(r"\w+", text)
    for word in words:
        close = get_close_matches(word, [s.replace("_", " ") for s in all_symptoms], n=1, cutoff=0.8)
        if close:
            for sym in all_symptoms:
                if sym.replace("_", " ") == close[0]:
                    extracted.append(sym)

    return list(set(extracted))

In [61]:
def predict_disease(symptoms_list):
    input_vector = np.zeros(len(symptoms_dict))

    for symptom in symptoms_list:
        if symptom in symptoms_dict:
            input_vector[symptoms_dict[symptom]] = 1
    input_df = pd.DataFrame([input_vector], columns=list(symptoms_dict.keys()))

    pred_proba = model.predict_proba(input_df)[0]
    pred_class = np.argmax(pred_proba)

    disease = le.inverse_transform([pred_class])[0]
    confidence = round(pred_proba[pred_class] * 100, 2)

    return disease, confidence, pred_proba

In [87]:
quotes = [
    "Your health is your greatest wealth.",
    "Every step toward wellness is a step toward a better you.",
    "Take care of your body; it’s the only place you have to live.",
    "Small healthy habits lead to big results.",
    "Healing is not a race; take it one day at a time.",
    "You deserve to feel your best.",
    "Listen to your body — it always speaks.",
    "A calm mind brings inner strength and self-confidence.",
    "Your health matters more than anything else.",
    "Wellness begins with small, consistent choices.",
    "Health is wealth, take care of yourself.",
    "A healthy outside starts from the inside.",
    "Every day is a chance to get stronger and healthier.",
    "Take a deep breath, your health matters the most.",
    "Remember, self-care is not selfish."
]
def liam():
    getSeverityDict()
    getDescription()
    getprecautionDict()

    print(" Welcome to HealthCare ChatBot. My Name is Liam")
    print("Hello! Please answer a few questions so I can understand your condition better.")


    name = input("What is your name? : ")
    age = input("Please enter your age: ")
    gender = input("What is your gender? (M/F/Other): ")

    symptoms_input = input("Describe your symptoms in a sentence (e.g., 'I have fever and stomach pain'): ")
    symptoms_list = extract_symptoms(symptoms_input, cols)

    if not symptoms_list:
        print("Sorry, I could not detect valid symptoms. Please try again with more details.")
        return

    print(f"Detected symptoms: {', '.join(symptoms_list)}")

    num_days = int(input("For how many days have you had these symptoms? : "))
    severity_scale = int(input("On a scale of 1–10, how severe do you feel your condition is? : "))
    pre_exist = input("Do you have any pre-existing conditions (e.g., diabetes, hypertension)? : ")
    lifestyle = input("Do you smoke, drink alcohol, or have irregular sleep? : ")
    family = input(" Any family history of similar illness? : ")

    disease, confidence, proba = predict_disease(symptoms_list)

    print("\n Let me ask you some more questions related to", disease)
    disease_symptoms = list(training[training['prognosis'] == disease].iloc[0][:-1].index[
        training[training['prognosis'] == disease].iloc[0][:-1] == 1
    ])

    asked = 0
    for sym in disease_symptoms:
        if sym not in symptoms_list and asked < 8:
            ans = input(f"Do you also have {sym.replace('_',' ')}? (yes/no): ").strip().lower()
            if ans == "yes":
                symptoms_list.append(sym)
            asked += 1

    disease, confidence, proba = predict_disease(symptoms_list)

    print("\n---------------- Result ----------------")
    print(f"Based on your answers, you may have **{disease}**")
    print(f"Confidence: {confidence}%")
    print(f"About: {description_list.get(disease, 'No description available.')}")

    if disease in precautionDictionary:
        print("\n Suggested precautions:")
        for i, prec in enumerate(precautionDictionary[disease], 1):
            print(f"{i}. {prec}")

    print("\n💡 " + random.choice(quotes))
    print("\nThank you for using the chatbot. Wishing you good health,", name + "!")

In [89]:
liam()

 Welcome to HealthCare ChatBot. My Name is Liam
Hello! Please answer a few questions so I can understand your condition better.
What is your name? : adi
Please enter your age: 23
What is your gender? (M/F/Other): m
Describe your symptoms in a sentence (e.g., 'I have fever and stomach pain'): stomach pain
Detected symptoms: stomach_pain
For how many days have you had these symptoms? : 2
On a scale of 1–10, how severe do you feel your condition is? : 7
Do you have any pre-existing conditions (e.g., diabetes, hypertension)? : no
Do you smoke, drink alcohol, or have irregular sleep? : irregular sleep
 Any family history of similar illness? : no

 Let me ask you some more questions related to Drug Reaction
Do you also have itching? (yes/no): no
Do you also have skin rash? (yes/no): no
Do you also have burning micturition? (yes/no): no
Do you also have spotting  urination? (yes/no): yes

---------------- Result ----------------
Based on your answers, you may have **Drug Reaction**
Confidence

In [91]:
liam()

 Welcome to HealthCare ChatBot. My Name is Liam
Hello! Please answer a few questions so I can understand your condition better.
What is your name? : adi
Please enter your age: 23
What is your gender? (M/F/Other): f
Describe your symptoms in a sentence (e.g., 'I have fever and stomach pain'): fatigue
Detected symptoms: fatigue
For how many days have you had these symptoms? : 2
On a scale of 1–10, how severe do you feel your condition is? : 7
Do you have any pre-existing conditions (e.g., diabetes, hypertension)? : no
Do you smoke, drink alcohol, or have irregular sleep? : no
 Any family history of similar illness? : no

 Let me ask you some more questions related to Varicose veins
Do you also have cramps? (yes/no): yes
Do you also have bruising? (yes/no): yes
Do you also have obesity? (yes/no): no
Do you also have swollen legs? (yes/no): no
Do you also have swollen blood vessels? (yes/no): no
Do you also have prominent veins on calf? (yes/no): no

---------------- Result ---------------

In [93]:
liam()

 Welcome to HealthCare ChatBot. My Name is Liam
Hello! Please answer a few questions so I can understand your condition better.
What is your name? : adi
Please enter your age: 23
What is your gender? (M/F/Other): m
Describe your symptoms in a sentence (e.g., 'I have fever and stomach pain'): fever and cold
Detected symptoms: fever, chills
For how many days have you had these symptoms? : 3
On a scale of 1–10, how severe do you feel your condition is? : 7
Do you have any pre-existing conditions (e.g., diabetes, hypertension)? : diabetes
Do you smoke, drink alcohol, or have irregular sleep? : irregular sleep
 Any family history of similar illness? : no

 Let me ask you some more questions related to Allergy
Do you also have continuous sneezing? (yes/no): yes
Do you also have shivering? (yes/no): yes
Do you also have watering from eyes? (yes/no): yes

---------------- Result ----------------
Based on your answers, you may have **Allergy**
Confidence: 100.0%
About: An allergy is an immune s